In [71]:
from ddgs import DDGS
import numpy as np
from langchain.messages import HumanMessage, SystemMessage
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

In [72]:
query="Which is the latest model released by Anthropic?"

In [73]:
response = DDGS().text(query, max_results=7, backend="auto")
response

[{'title': 'Claude (languagemodel) - Wikipedia',
  'href': 'https://en.wikipedia.org/wiki/Claude_(language_model)',
  'body': "Claude is a series of large languagemodelsdevelopedbyAnthropicand firstreleasedin 2023.On June 20, 2024,AnthropicreleasedClaude 3.5 Sonnet, which, according to the company's own benchmarks, performed better than the larger Claude 3 Opus."},
 {'title': "Claude 3.5 API:LatestInformation and Beginner's Tutorial!",
  'href': 'https://apidog.com/blog/claude-3-5-api/',
  'body': 'On June 21, 2024,AnthropicreleaseditslatestAImodel, Claude 3.5 Sonnet. In this article, we will summarizethelatestinformation about Claude 3.5 and provide an easy-to-understand tutorial on how to use the Claude 3.5 API.'},
 {'title': 'Introducing Claude 4 \\Anthropic',
  'href': 'https://www.anthropic.com/news/claude-4',
  'body': 'Claude Opus 4istheworld’s best codingmodel, with sustained performance on complex, long-running tasks and agent workflows. Claude Sonnet 4 is a significant upgrad

In [74]:
import trafilatura
import requests

def url_content(url, max_chars=2000, timeout=5):
    try:
        resp = requests.get(url, timeout=timeout, headers={"User-Agent": "Mozilla/5.0"})
        resp.raise_for_status()
        content = trafilatura.extract(resp.text)
        if content:
            return content[:max_chars]
    except Exception:
        pass
    return None

In [75]:
from concurrent.futures import ThreadPoolExecutor

def fetch_item(item):
    return {
        'title': item['title'],
        'body': item['body'],
        'content': url_content(item['href'])
    }

with ThreadPoolExecutor(max_workers=7) as executor:
    data = list(executor.map(fetch_item, response))
data

[{'title': 'Claude (languagemodel) - Wikipedia',
  'body': "Claude is a series of large languagemodelsdevelopedbyAnthropicand firstreleasedin 2023.On June 20, 2024,AnthropicreleasedClaude 3.5 Sonnet, which, according to the company's own benchmarks, performed better than the larger Claude 3 Opus.",
  'content': "Claude (language model)\n| Claude | |\n|---|---|\n| Developer | Anthropic |\n| Initial release | March 2023 |\n| Stable release | Claude Opus 4.6 / February 5, 2026 Claude Sonnet 4.6 / February 17, 2026 Claude Haiku 4.5 / October 15, 2025 |\n| Platform | Cloud computing platforms |\n| Type | |\n| License | Proprietary |\n| Website | claude |\nClaude is a series of large language models developed by Anthropic and first released in 2023. It is named after Claude Shannon, who pioneered information theory.\nClaude is used for software development via Claude Code.[1]\nThe use of Claude by US federal agencies is being phased out after Anthropic refused in February 2026 to remove cont

In [76]:
def chunk_text(text, chunk_size=500):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i+chunk_size])
    return chunks

In [77]:
from sentence_transformers import SentenceTransformer
x=0
if(x==1):
    model = SentenceTransformer("all-MiniLM-L6-v2")

In [78]:

chunks = []
for item in data:
    if item['content']:
        chunks.extend(chunk_text(item['content']))

chunk_embeddings = model.encode(chunks, batch_size=64, show_progress_bar=False)
query_embedding = model.encode(query)

In [79]:
similarities = np.dot(chunk_embeddings, query_embedding) / (
    np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(query_embedding)
)
top_indices = np.argsort(similarities)[-5:][::-1]
top_chunks = [chunks[i] for i in top_indices]

context = "\n\n".join(top_chunks)
context

"Anthropic's powerful Opus 4.1 model is here - how to access it (and why you'll want to)\nZDNET's key takeaways\n- Anthropic launched Claude Opus 4.1.\n- The model exceeds the predecessor's performance on complex tasks.\n- It is available to paid Claude users, Claude Code, API, Amazon Bedrock, and Google Cloud's Vertex AI.\nIn May, Anthropic released Claude Opus 4, which the company dubbed its most powerful model yet and the best coding model in the world. Only three months later, Anthropic is upping \n\nthe ante further by launching the highly anticipated Claude Opus 4.1, which now takes its predecessor's crown as Anthropic's most advanced model.\nThe Opus family of models is the company's most advanced, intelligent AI models geared toward tackling complex problems. As a result, Claude Opus 4.1, released on Tuesday, excels at those tasks and can even one-up its predecessor on agentic tasks, real-world coding, and reasoning, according to Anthropic.\nThe model also comes as the industry

In [80]:
def generate_answer(query, context):

    model = init_chat_model("groq:openai/gpt-oss-120b")
    conversation = [
        SystemMessage(
            "You are a helpful assistant. Use the context to answer the user question briefly."
        ),
        HumanMessage(
            f"""
                User Question: {query}

                Context: {context}
            """
        )
    ]

    return model.invoke(conversation).content

In [81]:
generate_answer(query,context)

'The most recent model Anthropic has released is **Claude\u202fSonnet\u202f4.5**.'